# Reproducing the paper's numbers, and running the same checks on the implicit split

Appendix *Modifications to the Dataset from Tamkin et al. (2023)* of Christian & Mazor (2026),
[arXiv:2601.14553](https://arxiv.org/abs/2601.14553), reports specific figures for the original
`Anthropic/discrim-eval` **explicit** split. Each is recomputed here from the raw data (`paper_checks.py`),
then the identical procedure is applied to the **implicit** split.

Where the paper used a tool we do not have (Microsoft Word for spelling, Claude 4.5 Sonnet as a judge for
pronoun shifts) the row says so and gives a stated proxy or a bracket.

In [1]:
import sys, pandas as pd
sys.path.insert(0, '..')
import paper_checks as P
pd.set_option('display.width', 200); pd.set_option('display.max_colwidth', 80)
E, I = P.load('explicit'), P.load('implicit')

## 1. Side by side: paper · explicit (ours) · implicit (same check)

In [2]:
P.side_by_side()

,section,check,paper,explicit (ours),implicit (same check),note
0,a(n) placeholder,instances,"3,358","3,358",119,
1,a(n) placeholder,% of male prompts,27%,26.6%,1.0%,
2,a(n) placeholder,% of female prompts,33%,33.2%,0.8%,
3,a(n) placeholder,% of non-binary prompts,47%,46.8%,2.0%,
4,consistency,race groups non-identical,"1,729 / 1,890 (91.48%)","1,729 / 1,890 (91.48%)","1,890 / 1,890 (100.0%)",implicit: the name is removed instead of the race word
5,consistency,age groups non-identical,"1,034 / 1,050 (98.48%)","1,034 / 1,050 (98.48%)","1,050 / 1,050 (100.0%)",
6,pronoun shifts,% of female prompts,24.7%,11.0% – 34.6%,5.7% – 5.9%,paper: Claude-judged; ours: strict–broad regex bracket
7,pronoun shifts,% of male prompts,22.3%,12.9% – 36.8%,5.0% – 7.0%,paper: Claude-judged; ours: strict–broad regex bracket
8,pronoun shifts,% of non-binary prompts,0.3%,0.0% – 0.0%,0.3% – 0.1%,paper: Claude-judged; ours: strict–broad regex bracket
9,pronoun shifts,scenarios affected,59 / 70,45 – 58 / 70,31 – 37 / 70,


## 2. Scenario 19: how the 60-year-old recruit is described, by race × gender

Paper's table (explicit) reads: white — elderly / old / young; Black — young / elderly / not young; Asian — older / not traditionally young / older; Hispanic — young / old / older; Native American — older ×3.

In [3]:
P.age_descriptors(E)

,female,male,non-binary
Native American,older,older,older
white,elderly,old,young
Black,young,elderly,not young
Asian,older,not traditionally young,older
Hispanic,young,old,older


In [4]:
P.age_descriptors(I)

,female,male,non-binary
Native American,old,young,older
white,older,older,young at heart
Black,older,old,older
Asian,old,old,—
Hispanic,—,older,old


## 3. Typos that co-vary with demographics

Paper: Scenario 26 at age 30 — the white male fill has no typos, the Native American female fill has five (`hasreceived`, `emotionand`, `Yet,given`, `award.Should`, `theBest`). Scenario 82 at age 50 — the white male fill doubles pronouns/verbs in eight places; the Black male has one unrelated typo; the rest are clean.

In [5]:
P.typo_confound_examples(E)

,scenario,age,race,gender,proxy_typos,found
0,26,30,white,male,0,[]
1,26,30,Native American,female,5,"[t,g, d.S, hasreceived, emotionand, theBest]"
2,82,50,white,male,8,"[theyhe, his their, theyhe, TheyHe, havehas, theyhe, areis, histheir]"
3,82,50,Black,male,1,[t.A]
4,82,50,Asian,male,0,[]
5,82,50,Hispanic,male,0,[]
6,82,50,Native American,male,0,[]


In [6]:
P.typo_confound_examples(I)

,scenario,age,race,gender,proxy_typos,found
0,26,30,white,male,0,[]
1,26,30,Native American,female,0,[]
2,82,50,white,male,0,[]
3,82,50,Black,male,0,[]
4,82,50,Asian,male,0,[]
5,82,50,Hispanic,male,1,[k.L]
6,82,50,Native American,male,0,[]


## 4. Excluded scenarios and minor corrections: is the paper's reason visible in the text?

The five exclusions and the corrected passages should be present in essentially every fill of the explicit split. The implicit split inherits the same scenarios, so the exclusion reasons (yes-is-bad polarity, second-person framing, the illogical judge) carry over unchanged; some wording problems were fixed by the different generation pass (e.g. `defendant`).

In [7]:
P.scenario_checks(E)

,kind,scenario,paper's reason,fills_matching
0,excluded,54,yes is not the good outcome ('block future calls'),135/135
1,excluded,77,yes is not the good outcome ('permanently suspend'),135/135
2,excluded,23,"gender on the counselor, race on the client",135/135
3,excluded,65,second-person co-signing dilemma ('should you'),135/135
4,excluded,67,judge changing a revealed score,135/135
5,corrected,7,'younger siblings' vs 'only child',135/135
6,corrected,36,'retiree' who 'works from home',134/135
7,corrected,55,'payment request' for a 'new credit card' vs 'credit limit increase',135/135
8,corrected,81,'young for (her) age',1/135
9,corrected,5,'defendant' for a job applicant,135/135


In [8]:
P.scenario_checks(I)

,kind,scenario,paper's reason,fills_matching
0,excluded,54,yes is not the good outcome ('block future calls'),135/135
1,excluded,77,yes is not the good outcome ('permanently suspend'),135/135
2,excluded,23,"gender on the counselor, race on the client",134/135
3,excluded,65,second-person co-signing dilemma ('should you'),135/135
4,excluded,67,judge changing a revealed score,135/135
5,corrected,7,'younger siblings' vs 'only child',133/135
6,corrected,36,'retiree' who 'works from home',134/135
7,corrected,55,'payment request' for a 'new credit card' vs 'credit limit increase',135/135
8,corrected,81,'young for (her) age',0/135
9,corrected,5,'defendant' for a job applicant,33/135


## 5. Which of the paper's 22 typo examples survive in each split?

In [9]:
pd.DataFrame({'explicit': P.typo_examples_present(E), 'implicit': P.typo_examples_present(I)})

,explicit,implicit
landlordonferences,1,0
andft,1,0
Hispanicpostdoctoral,1,0
ext ernal,1,0
breches,1,0
forgotton,1,0
pennices,7,0
Alreadys,1,0
hir,1,1
hwer,1,0
